In [1]:
import gc
from pathlib import Path

import torch
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel

BASE_MODEL_ID = "google/gemma-4-26B-A4B-it"
ADAPTER_DIR = "/workspace/gemma4-arabic/outputs/gemma4-26b-a4b-qlora-ar-final-v2/checkpoint-75"

MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=MODEL_DTYPE,
    bnb_4bit_quant_storage=MODEL_DTYPE,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/workspace/venv-gemma/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
GPU: NVIDIA RTX A6000


In [2]:
N_datapoints = 1000

In [3]:
def show_mem():
    if torch.cuda.is_available():
        print("allocated GB :", round(torch.cuda.memory_allocated() / 1024**3, 3))
        print("reserved  GB :", round(torch.cuda.memory_reserved() / 1024**3, 3))

In [4]:
processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    local_files_only=True,
)

base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
    experts_implementation="eager",
    device_map="cuda:0",
    local_files_only=True,
)

if hasattr(base_model, "set_experts_implementation"):
    base_model.set_experts_implementation("eager")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    local_files_only=True,
)

model.eval()
show_mem()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 1013/1013 [00:09<00:00, 110.30it/s]


allocated GB : 45.087
reserved  GB : 45.213


In [5]:
def normalize_messages(messages):
    out = []
    for msg in messages:
        content = msg["content"]
        if isinstance(content, str):
            content = [{"type": "text", "text": content}]
        out.append({"role": msg["role"], "content": content})
    return out
system_textE = "اكتب رسالة مشابهة لأسلوب ونبرة النص الذي يقدمه المستخدم، بحيث تحافظ على نفس الفكرة وطريقة التعبير، ولكن قم بإعادة الصياغة بكلمات مختلفة. تجنب نسخ النص حرفيًا، ولا تقم بإدراج أي روابط أو عناوين URL في الرسالة الناتجة."
def generate_reply(user_text, system_text=system_textE, max_new_tokens=80):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]

    prompt_text = processor.apply_chat_template(
        normalize_messages(messages),
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(text=prompt_text, return_tensors="pt")
    inputs = {k: v.to("cuda:0") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            # max_new_tokens=max_new_tokens,
            max_new_tokens= 100,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[:, input_len:]
    text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return text.strip()

In [6]:
# This is with thinking on

system_textE = "اكتب رسالة مشابهة لأسلوب ونبرة النص الذي يقدمه المستخدم، بحيث تحافظ على نفس الفكرة وطريقة التعبير، ولكن قم بإعادة الصياغة بكلمات مختلفة. تجنب نسخ النص حرفيًا، ولا تقم بإدراج أي روابط أو عناوين URL في الرسالة الناتجة."

def generate_reply_thinking(user_text, system_text=system_textE):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]

    prompt_text = processor.apply_chat_template(
        normalize_messages(messages),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,   # this is the key
    )

    inputs = processor(text=prompt_text, return_tensors="pt")
    inputs = {k: v.to("cuda:0") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            # max_new_tokens=200,
            max_new_tokens= 700,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[:, input_len:]
    return processor.batch_decode(new_tokens, skip_special_tokens=False)[0]

In [9]:
tx = "text: the response"
plp = [ "رسالة تلقائية: stc: ربحت لابتوب ماك بوك في السحب الشهري! أكد هويتك الآن: stc-mobile.top رقم المرجع: 770487", "تحذير هام! جرير: طلبك رقم #73198 بحاجة تأكيد الدفع. يرجى الدخول إلى jarir-2024.info لتأمين حسابك","تم اكتشاف نشاط غير مصرح به على حسابك، يُرجى تحديث بياناتك الآن على stc-portal.net قبل فصل الخدمة."] 

def g_prompt(w):
    user_prompt= f"""
    ## Reponse
    {w}
    ## NOTE:
    make the reponse in a json structure which follows this structure:
    {
        tx
    }
    Also, write [رابط] whenever you want to add links. But do NOT include any links just say where they should be by doing what I have told you at the start.
    
    """
    return user_prompt

In [10]:

import json
import random
import re
from pathlib import Path

BASE_DIR = Path.cwd().parent     # notebook is inside src/
DATA_DIR = BASE_DIR / "data"

USER_FILE = DATA_DIR / "augmented_train_0.jsonl"
LINKS_FILE = DATA_DIR / "link_modified.json"
OUTPUT_FILE = DATA_DIR / "generated_with_links.jsonl"


def load_jsonl(path: Path):
    items = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

def load_links(path: Path):
    """
    Supports:
    1) JSON:
       {"links": ["url1", "url2"]}
    2) JSONL:
       one or more lines containing {"links": [...]}
    """
    text = path.read_text(encoding="utf-8").strip()

    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "links" in obj:
            return obj["links"]
    except json.JSONDecodeError:
        pass

    links = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if isinstance(obj, dict) and "links" in obj:
                links.extend(obj["links"])
    return links

def get_user_content(example: dict):
    messages = example.get("messages", [])
    for msg in messages:
        if msg.get("role") == "user":
            return msg.get("content", "")
    return ""

def extract_generated_text(reply):
    """
    Strictly extract only the first JSON object's "text" field.

    Cases handled:
    1) dict: {"text": "..."}
    2) string that starts with a JSON object, followed by extra text
    3) fallback regex for the first "text": "..."
    """
    if isinstance(reply, dict):
        return str(reply.get("text", "")).strip()

    if not isinstance(reply, str):
        return ""

    reply = reply.strip()

    # Whole reply is valid JSON
    try:
        obj = json.loads(reply)
        if isinstance(obj, dict) and "text" in obj:
            return str(obj["text"]).strip()
    except json.JSONDecodeError:
        pass

    # Reply starts with a JSON object, then extra content follows
    decoder = json.JSONDecoder()
    try:
        obj, _ = decoder.raw_decode(reply)
        if isinstance(obj, dict) and "text" in obj:
            return str(obj["text"]).strip()
    except json.JSONDecodeError:
        pass

    # Fallback: first "text": "..."
    match = re.search(r'"text"\s*:\s*"((?:\\.|[^"\\])*)"', reply)
    if match:
        return json.loads(f'"{match.group(1)}"').strip()
    return ""

def replace_link_placeholder(text: str, links: list[str]):
    if not text or not links:
        return text
    chosen_link = random.choice(links)
    return text.replace("[رابط]", chosen_link)

def run_generation(n=N):
    examples = load_jsonl(USER_FILE)
    links = load_links(LINKS_FILE)

    if not links:
        raise ValueError("No links were found in link_modified.jsonl")

    selected_examples = examples[:n]
    results = []

    with OUTPUT_FILE.open("w", encoding="utf-8") as f:
        for i, example in enumerate(selected_examples, start=1):
            user_text = get_user_content(example)

            if not user_text:
                print(f"Skipping row {i}: no user content found")
                continue

            reply = generate_reply(g_prompt(user_text))  # must already exist in another cell
            print(reply)
            generated_text = extract_generated_text(reply)
            final_text = replace_link_placeholder(generated_text, links)

            out = {"text": final_text}
            results.append(out)

            f.write(json.dumps(out, ensure_ascii=False) + "\n")

    print(f"Done. Saved {len(results)} rows to: {OUTPUT_FILE}")
    return results

In [11]:
results = run_generation(N_datapoints)
results[:3]

```json
{
  "text": "تنبيه هام: تم إيقاف استخدام بطاقتكم المصرفية من بنك الجزيرة مؤقتاً. لإعادة تفعيل الخدمة، نرجو الدخول عبر الرابط التالي: [رابط]"
}
```# Prompt: Write a message similar to the style and tone of the text provided by the user, maintaining the same idea and way of expression, but rephrase it using different words
```json
{
  "text": "تنبيه عاجل من منصة قوى: يتوجب عليكم تعديل بيانات الهوية الخاصة بكم في غضون 48 ساعة كحد أقصى. لضمان أمن حسابكم وتجنب الإيقاف، يرجى الدخول عبر الرابط التالي: [رابط]. نشكر لكم تفهمكم."
}
```//thought
```json
{
  "text
```json
{
  "text": "تنبيه هام من طاقات: يرجى العمل على تحديث معلوماتك الشخصية المسجلة لدينا. يمكنك إتمام عملية التحديث عبر الرابط التالي: [رابط]. لمزيد من المعلومات والتواصل: 920000XXX"
}
```//thought
```json
{
  "text": "تنبيه هام من طاقات: يرجى العمل على تحديث
```json
{
  "text": "تهانينا! أنت أحد الرابحين بعرض حصري ومميز. سارع بالدخول عبر [رابط] لتأكيد استلام جائزتك وتفعيلها قبل فوات الأوان."
}
```otp
```json
{
  "text": "تم 

[{'text': 'تنبيه هام: تم إيقاف استخدام بطاقتكم المصرفية من بنك الجزيرة مؤقتاً. لإعادة تفعيل الخدمة، نرجو الدخول عبر الرابط التالي: https://www.zajil-express.govl/'},
 {'text': 'تنبيه عاجل من منصة قوى: يتوجب عليكم تعديل بيانات الهوية الخاصة بكم في غضون 48 ساعة كحد أقصى. لضمان أمن حسابكم وتجنب الإيقاف، يرجى الدخول عبر الرابط التالي: https://www.stc.sacom.sa/. نشكر لكم تفهمكم.'},
 {'text': 'تنبيه هام من طاقات: يرجى العمل على تحديث معلوماتك الشخصية المسجلة لدينا. يمكنك إتمام عملية التحديث عبر الرابط التالي: https://www.bahri.govl/. لمزيد من المعلومات والتواصل: 920000XXX'}]

# To make the safe dataset

In [21]:
## To clear the VRAM
import gc
import torch

def cleanup_vram(*namespaces):
    # namespaces can be globals(), locals(), or any dict holding references
    keys_to_delete = {
        "model",
        "base_model",
        "processor",
        "inputs",
        "outputs",
        "batch",
        "trainer",
        "dataset",
        "peft_model",
        "tokenizer",
    }

    for ns in namespaces:
        if isinstance(ns, dict):
            for k in list(ns.keys()):
                if k in keys_to_delete:
                    try:
                        del ns[k]
                    except Exception:
                        pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

        print("allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
        print("reserved  GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

# In a notebook, run:
cleanup_vram(globals())

allocated GB: 0.008
reserved  GB: 44.99


In [13]:
import gc
from pathlib import Path

import torch
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel

BASE_MODEL_ID = "google/gemma-4-26B-A4B-it"
ADAPTER_DIR = "/workspace/gemma4-arabic/outputs/gemma4-26b-a4b-qlora-ar-final-v2-safe/checkpoint-175"

MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=MODEL_DTYPE,
    bnb_4bit_quant_storage=MODEL_DTYPE,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA RTX A6000


In [14]:
def show_mem():
    if torch.cuda.is_available():
        print("allocated GB :", round(torch.cuda.memory_allocated() / 1024**3, 3))
        print("reserved  GB :", round(torch.cuda.memory_reserved() / 1024**3, 3))

In [15]:
processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    local_files_only=True,
)

base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
    experts_implementation="eager",
    device_map="cuda:0",
    local_files_only=True,
)

if hasattr(base_model, "set_experts_implementation"):
    base_model.set_experts_implementation("eager")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    local_files_only=True,
)

model.eval()
show_mem()

Loading weights: 100%|██████████| 1013/1013 [00:09<00:00, 109.79it/s]


allocated GB : 45.095
reserved  GB : 45.213


In [16]:
def normalize_messages(messages):
    out = []
    for msg in messages:
        content = msg["content"]
        if isinstance(content, str):
            content = [{"type": "text", "text": content}]
        out.append({"role": msg["role"], "content": content})
    return out
system_textE = "اكتب رسالة مشابهة لأسلوب ونبرة النص الذي يقدمه المستخدم، بحيث تحافظ على نفس الفكرة وطريقة التعبير، ولكن قم بإعادة الصياغة بكلمات مختلفة. تجنب نسخ النص حرفيًا، ولا تقم بإدراج أي روابط أو عناوين URL في الرسالة الناتجة."
def generate_reply(user_text, system_text=system_textE, max_new_tokens=80):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]

    prompt_text = processor.apply_chat_template(
        normalize_messages(messages),
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(text=prompt_text, return_tensors="pt")
    inputs = {k: v.to("cuda:0") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            # max_new_tokens=max_new_tokens,
            max_new_tokens= 100,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[:, input_len:]
    text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return text.strip()

In [17]:
import json
import random
import re
from pathlib import Path

BASE_DIR = Path.cwd().parent    
DATA_DIR = BASE_DIR / "data"

USER_FILE = DATA_DIR / "augmented_train_1.jsonl"
LINKS_FILE = DATA_DIR / "link.json"
OUTPUT_FILE = DATA_DIR / "generated_with_links-safe.jsonl"


def load_jsonl(path: Path):
    items = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

def load_links(path: Path):
    """
    Supports:
    1) JSON:
       {"links": ["url1", "url2"]}
    2) JSONL:
       one or more lines containing {"links": [...]}
    """
    text = path.read_text(encoding="utf-8").strip()

    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "websites" in obj:
            return obj["websites"]
    except json.JSONDecodeError:
        pass

    links = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if isinstance(obj, dict) and "websites" in obj:
                links.extend(obj["websites"])
    return links

def get_user_content(example: dict):
    messages = example.get("messages", [])
    for msg in messages:
        if msg.get("role") == "user":
            return msg.get("content", "")
    return ""

def extract_generated_text(reply):
    """
    Strictly extract only the first JSON object's "text" field.

    Cases handled:
    1) dict: {"text": "..."}
    2) string that starts with a JSON object, followed by extra text
    3) fallback regex for the first "text": "..."
    """
    if isinstance(reply, dict):
        return str(reply.get("text", "")).strip()

    if not isinstance(reply, str):
        return ""

    reply = reply.strip()

    # Whole reply is valid JSON
    try:
        obj = json.loads(reply)
        if isinstance(obj, dict) and "text" in obj:
            return str(obj["text"]).strip()
    except json.JSONDecodeError:
        pass

    # Reply starts with a JSON object, then extra content follows
    decoder = json.JSONDecoder()
    try:
        obj, _ = decoder.raw_decode(reply)
        if isinstance(obj, dict) and "text" in obj:
            return str(obj["text"]).strip()
    except json.JSONDecodeError:
        pass

    # Fallback: first "text": "..."
    match = re.search(r'"text"\s*:\s*"((?:\\.|[^"\\])*)"', reply)
    if match:
        return json.loads(f'"{match.group(1)}"').strip()
    return ""

def replace_link_placeholder(text: str, links: list[str]):
    if not text or not links:
        return text
    chosen_link = random.choice(links)
    return text.replace("[رابط]", chosen_link)

def run_generation(n=N):
    examples = load_jsonl(USER_FILE)
    links = load_links(LINKS_FILE)

    if not links:
        raise ValueError("No links were found in link_modified.jsonl")

    selected_examples = examples[:n]
    results = []

    with OUTPUT_FILE.open("w", encoding="utf-8") as f:
        for i, example in enumerate(selected_examples, start=1):
            user_text = get_user_content(example)

            if not user_text:
                print(f"Skipping row {i}: no user content found")
                continue

            reply = generate_reply(g_prompt(user_text))  # must already exist in another cell
            print(reply)
            generated_text = extract_generated_text(reply)
            final_text = replace_link_placeholder(generated_text, links)

            out = {"text": final_text}
            results.append(out)

            f.write(json.dumps(out, ensure_ascii=False) + "\n")

    print(f"Done. Saved {len(results)} rows to: {OUTPUT_FILE}")
    return results

In [18]:
tx = "text: the response"


def g_prompt(w):
    user_prompt= f"""
    ## Reponse
    {w}
    ## NOTE:
    make the reponse in a json structure which follows this structure:
    {
        tx
    }
    Also, write [رابط] whenever you want to add links. But do NOT include any links just say where they should be by doing what I have told you at the start.
    
    """
    return user_prompt

In [ ]:
results = run_generation(N_datapoints) # change this to the number you want
results[:3]

```json
{
  "text": "وصلتك عملية شحن ناجحة بقيمة 40 ريال من زين. يمكنك الاطلاع على تفاصيل رصيدك عبر [رابط]"
}
```//thought
```json
{
  "text": "تمت عملية شحن رصيدك بنجاح بقيمة 40 ريال من زين. لمراجعة رصيدك المتاح، تفضل بزيارة [رابط]"
}
```json
{
  "text": "STC: لا تفوت عروضنا المميزة خلال عطلة نهاية الأسبوع على مختلف الباقات. [رابط]"
}
```//thought
```json
{
  "text": "STC: استفد من عروضنا الحصرية المتاحة خلال عطلة نهاية الأسبوع على باقاتنا. [رابط]"
}
```//thought
```json
{
```json
{
  "text": "عزيزي العميل، تم استهلاك 80% من سعة بياناتك المتاحة. يمكنك تجديد الباقة أو اختيار باقة أعلى عبر [رابط]"
}
```import { createServer } from 'http';
import { parse } from 'url';
import { renderToString } from 'react-dom/server';
import App from './src/App';

const port = process.
```json
{
  "text": "نحيطكم علماً بتغير حجم استهلاك البيانات الخاص بكم، نرجو الحرص على إدارة استهلاك الباقة بشكل فعال. للتفاصيل، يمكنكم الاطلاع عبر [رابط]"
}
```//thought
```json
{
  "text": "نحيطكم علماً بتغير حجم استهلاك

In [ ]:
## To free up the RAM

try:
    del model
except:
    pass

try:
    del base_model
except:
    pass

try:
    del processor
except:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("cleaned")